# 第11章 可转换债券 — 编程实验完整解答

[![Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/albertandking/fixed-income/blob/main/notebooks/solutions/ch11_solutions.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/albertandking/fixed-income/main?labpath=notebooks/solutions/ch11_solutions.ipynb)

本 notebook 给出本章全部编程实验的完整可运行解答；联网（akshare）部分以注释/降级方式给出，离线也能跑通。


In [ ]:
# 自举单元：Colab/Binder 自动安装 fi；本地跳过。
import importlib.util, sys, subprocess
if importlib.util.find_spec('fi') is None:
    if 'google.colab' in sys.modules:
        subprocess.run(['git','clone','--depth','1','https://github.com/albertandking/fixed-income.git','/content/fi-book'],check=False)
        subprocess.run([sys.executable,'-m','pip','install','-e','/content/fi-book'],check=False)
    else:
        print('提示：仓库根目录执行 `uv sync --extra all` 后运行本 notebook。')


## 编程实验 6：例11.1/11.2 + 图11-1（股债性切换）


In [ ]:
import numpy as np
from fi import convertible as cb, plotting
plotting.use_chinese_style()
floor = cb.bond_floor(100, 0.015, 5, 0.05); print('纯债底=', round(floor,2))
for S in (5,8,12):
    px = cb.price_convertible(S,0.30,0.03,5,100,0.015,10,n_steps=200)['price']
    print(f'S={S}: 可转债={px:.2f} 转股价值={10*S} 转股溢价率={(px-10*S)/(10*S)*100:.1f}%')
s = np.linspace(2,16,36)
fig, ax = plotting.new_axes()
ax.plot(s, [cb.price_convertible(si,0.30,0.03,5,100,0.015,10,n_steps=120)['price'] for si in s], lw=2, label='可转债')
ax.plot(s, 10*s, '--', label='转股价值'); ax.axhline(floor, ls=':', color='gray', label=f'纯债底{floor:.0f}')
ax.set_xlabel('正股价格 S'); ax.set_ylabel('价值'); ax.set_title('股债性切换'); ax.legend(); fig.tight_layout()


## 编程实验 7：强赎封顶股性收益


In [ ]:
for S in (10,12,14,16):
    no = cb.price_convertible(S,0.30,0.03,5,100,0.015,10,n_steps=200)['price']
    wc = cb.price_convertible(S,0.30,0.03,5,100,0.015,10,n_steps=200,call_price=105)['price']
    print(f'S={S}: 无强赎={no:.2f} 有强赎={wc:.2f} 差={no-wc:.2f}')
print('高股价区强赎封顶股性收益')


## 编程实验 8：fi vs QuantLib 信用利差敏感性


In [ ]:
import QuantLib as ql
today = ql.Date(15,6,2026); ql.Settings.instance().evaluationDate = today
dc, cal = ql.Actual365Fixed(), ql.NullCalendar()
spot = ql.QuoteHandle(ql.SimpleQuote(8.0))
rTS = ql.YieldTermStructureHandle(ql.FlatForward(today,0.03,dc)); qTS = ql.YieldTermStructureHandle(ql.FlatForward(today,0.0,dc))
volTS = ql.BlackVolTermStructureHandle(ql.BlackConstantVol(today,cal,0.30,dc))
proc = ql.BlackScholesMertonProcess(spot,qTS,rTS,volTS)
ex = ql.AmericanExercise(today, today+ql.Period(5,ql.Years))
sched = ql.Schedule(today, today+ql.Period(5,ql.Years), ql.Period(ql.Annual), cal, ql.Unadjusted, ql.Unadjusted, ql.DateGeneration.Backward, False)
bond = ql.ConvertibleFixedCouponBond(ex, 10.0, ql.CallabilitySchedule(), today, 0, [0.015], dc, sched, 100.0)
print(f'fi(无风险折现) = {cb.price_convertible(8,0.30,0.03,5,100,0.015,10,n_steps=200)["price"]:.3f}')
for cs in (0.0,0.02,0.04):
    bond.setPricingEngine(ql.BinomialConvertibleEngine(proc,'crr',200, ql.QuoteHandle(ql.SimpleQuote(cs)), ql.DividendSchedule()))
    print(f'QuantLib 信用利差={cs*100:.0f}%: NPV={bond.NPV():.3f}')
